# 勇者傳說 — Environment Generator (T4/L4 GPU, Self-Contained)

Generate seamless tiles, battle backgrounds, and interior scenes.

| Category | Model | VRAM | Speed | Count |
|----------|-------|------|-------|-------|
| Tiles (seamless) | SDXL + LoRA | ~8 GB | ~10-15s/tile | 12 |
| Battle BGs | SDXL | ~7 GB | ~8s/img | 36 |
| Interiors | SDXL | ~7 GB | ~8s/img | 6 |

**Self-contained**: No repo clone needed.  
**Output**: Google Drive `/MyDrive/ai-rpg-game/outputs/`  
**Resume**: 中斷後重跑自動跳過已完成的

## 1. Setup

In [ ]:
# ── Install packages ──
# After this cell runs, the runtime will auto-restart.
# When it restarts, skip this cell and run from the next one.
!pip install -q "numpy<2" scipy  # MUST be first — fixes numpy/scipy C-extension mismatch
!pip install -q diffusers transformers accelerate safetensors "Pillow>=10,<12" matplotlib

# Auto-restart runtime so the new numpy is loaded into memory
import os
os.kill(os.getpid(), 9)

In [ ]:
# ── Imports & Setup (run this after runtime restart) ──
import gc, json, os, time
from pathlib import Path
import numpy as np
from PIL import Image

print(f'numpy version: {np.__version__}')  # should be 1.x

try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

GDRIVE_BASE = Path('/content/drive/MyDrive/ai-rpg-game')
OUTPUT_BASE = GDRIVE_BASE / 'outputs' if ON_COLAB else Path('outputs')
MODELS_DIR = GDRIVE_BASE / 'models' if ON_COLAB else Path('models')

for d in [OUTPUT_BASE, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

import torch
if torch.cuda.is_available():
    DEVICE = 'cuda'
    DTYPE = torch.float16
    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = (getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)) / 1024**3
    print(f'GPU: {name} ({vram:.1f} GB VRAM)')
else:
    DEVICE = 'cpu'
    DTYPE = torch.bfloat16
    print('WARNING: No GPU!')

print(f'Output: {OUTPUT_BASE}')

## 2. Utilities

In [ ]:
class ProgressTracker:
    def __init__(self, task_name, output_dir):
        self.file = Path(output_dir) / f'_progress_{task_name}.json'
        self.completed = set()
        self.start_time = time.time()
        if self.file.exists():
            try:
                data = json.loads(self.file.read_text())
                self.completed = set(data.get('completed', []))
                print(f'[resume] {len(self.completed)} items already done')
            except Exception:
                pass

    def is_done(self, name): return name in self.completed

    def mark_done(self, name):
        self.completed.add(name)
        self.file.parent.mkdir(parents=True, exist_ok=True)
        self.file.write_text(json.dumps({
            'completed': sorted(self.completed),
            'count': len(self.completed),
            'last_updated': time.strftime('%Y-%m-%d %H:%M:%S'),
        }, indent=2))

    def summary(self, total):
        done = len(self.completed)
        elapsed = time.time() - self.start_time
        if done > 0:
            remaining = (total - done) * (elapsed / done)
            return f'{done}/{total} done, ~{remaining/60:.0f} min remaining'
        return f'0/{total} done'


def free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def print_vram():
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated(0) / 1024**3
        props = torch.cuda.get_device_properties(0)
        total = (getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)) / 1024**3
        print(f'[vram] {used:.1f}/{total:.1f} GB')

## 3. Prompt Data (Embedded)

In [ ]:
PROMPTS = {
    "_meta": {
        "negative_prompt_xl": "blurry, 3d render, photorealistic, photograph, smooth gradient, text, watermark, signature, modern, sci-fi, futuristic, complex background, busy background, multiple creatures, two creatures, pair, duplicate, group, crowd, many, low quality, worst quality, jpeg artifacts, deformed, ugly, mutation, extra limbs"
    },
    "tiles": [
        {"name": "tile_grass", "prompt_xl": "lush green grass tile, scattered tiny wildflowers, clover patches, rich soil visible between blades, varied green tones, natural meadow texture"},
        {"name": "tile_stone", "prompt_xl": "medieval cobblestone path tile, worn irregular grey stones, green moss growing between cracks, weathered texture, damp surface highlights"},
        {"name": "tile_wood", "prompt_xl": "dark oak wooden floor tile, natural wood grain pattern, plank seams visible, warm brown tones, polished surface with subtle highlights"},
        {"name": "tile_water", "prompt_xl": "clear blue water surface tile, gentle concentric ripples, light reflections, transparent depth effect, varied blue-teal tones"},
        {"name": "tile_sand", "prompt_xl": "desert sand ground tile, fine golden sand texture, scattered small pebbles and shell fragments, wind-swept patterns, warm tan tones"},
        {"name": "tile_dirt", "prompt_xl": "packed dirt path tile, compressed brown earth, embedded small stones, dried mud cracks, worn footpath texture"},
        {"name": "tile_snow", "prompt_xl": "fresh snow ground tile, pristine white snow surface, tiny frost crystal sparkles, subtle blue shadow in compressed areas, winter texture"},
        {"name": "tile_lava", "prompt_xl": "molten lava tile, glowing bright orange-yellow magma, dark cooled volcanic crust with cracks, heat distortion glow, volcanic texture"},
        {"name": "tile_dark_stone", "prompt_xl": "dark dungeon stone floor tile, polished black obsidian surface, glowing purple-violet magical cracks, sinister energy seeping through"},
        {"name": "tile_cave", "prompt_xl": "rough cave floor tile, uneven grey-brown rock surface, sparkling mineral crystal veins, damp patches, stalactite drip marks"},
        {"name": "tile_wall_stone", "prompt_xl": "medieval castle wall tile, large grey stone blocks, mortar between bricks, weathered fortress wall, arrow slit texture"},
        {"name": "tile_wall_wood", "prompt_xl": "tudor timber frame wall tile, dark wooden beams crossing white plaster, half-timbered construction, medieval European style"}
    ],
    "battle_backgrounds": [
        {"name": "battle_bg_r1", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG battle background, lush green rolling meadow hills, bright blue sky with fluffy white clouds, scattered colorful wildflowers, distant medieval village, warm sunlight, Secret of Mana style, vibrant pixel art landscape"},
        {"name": "battle_bg_r1_boss", "width": 1024, "height": 768, "prompt_xl": "epic JRPG boss battle background, dramatic hilltop arena, dark storm clouds swirling overhead, lightning bolts illuminating the scene, windswept tall grass, ominous purple-grey sky, tension-filled atmosphere, final confrontation setting"},
        {"name": "battle_bg_cave_r1", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG cave dungeon battle background, dark rough stone walls with hanging stalactites, luminous blue-green crystal formations providing eerie light, underground river reflecting cave ceiling, damp atmosphere, mysterious underground"},
        {"name": "battle_bg_r2", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG battle background, enchanted ancient forest, massive gnarled trees with thick canopy filtering golden dappled sunlight, mystical glowing particles floating in air, mossy forest floor, ethereal fairy-tale atmosphere, rich green tones"},
        {"name": "battle_bg_r2_boss", "width": 1024, "height": 768, "prompt_xl": "dark JRPG boss battle background, deeply corrupted ancient forest, twisted blackened trees with thorny branches, thick purple toxic fog rolling across ground, sinister glowing eyes in shadowy undergrowth, decaying vegetation, oppressive dark green-purple atmosphere"},
        {"name": "battle_bg_cave_r2", "width": 1024, "height": 768, "prompt_xl": "JRPG underground cave battle background, massive tree root system forming natural cavern, bioluminescent mushrooms and fungi providing soft blue-green glow, underground forest with moss and ferns, dripping water, mystical subterranean grove"},
        {"name": "battle_bg_r3", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG battle background, beautiful tropical beach, swaying palm trees, crystal clear turquoise ocean with gentle waves, white sand beach, colorful coral reef visible through clear water, bright sunny day, paradise setting"},
        {"name": "battle_bg_r3_boss", "width": 1024, "height": 768, "prompt_xl": "epic JRPG boss battle background, raging stormy ocean with towering dark waves, massive swirling whirlpool, jagged rocky outcrops, lightning splitting dark sky, sea foam and spray, treacherous deep sea battle arena, dramatic nautical confrontation"},
        {"name": "battle_bg_cave_r3", "width": 1024, "height": 768, "prompt_xl": "JRPG underwater grotto battle background, submerged cave with beautiful coral formations in pinks and oranges, shafts of golden light streaming from above, scattered treasure chests and gold coins on sandy floor, schools of fish, magical underwater atmosphere"},
        {"name": "battle_bg_r4", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG battle background, vast golden savanna plains stretching to horizon, silhouetted acacia trees, dramatic orange-gold sunset sky, tall swaying grass, distant mountains, warm African-inspired fantasy landscape, golden hour lighting"},
        {"name": "battle_bg_r4_boss", "width": 1024, "height": 768, "prompt_xl": "JRPG boss battle background, desolate volcanic wasteland, deeply cracked scorched earth, pools of bubbling molten lava glowing orange, thick ash clouds in sky, fiery red-orange sky, dead blackened trees, intense heat distortion, apocalyptic wasteland"},
        {"name": "battle_bg_cave_r4", "width": 1024, "height": 768, "prompt_xl": "JRPG underground cave battle background, ancient Egyptian-inspired temple ruins, massive carved stone pillars with hieroglyphs, golden torchlight flickering, sand drifts partially burying structures, mysterious artifacts, archaeological dungeon atmosphere"},
        {"name": "battle_bg_r5", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG battle background, majestic snow-covered mountain pass, frosted pine trees laden with snow, spectacular aurora borealis dancing in deep blue sky, towering icy mountain peaks, pristine white snow, cold winter atmosphere, Nordic fantasy landscape"},
        {"name": "battle_bg_r5_boss", "width": 1024, "height": 768, "prompt_xl": "JRPG boss battle background, ancient frozen throne room carved from glacier ice, massive translucent blue ice pillars, frozen crystalline floor reflecting dim light, howling blizzard visible through shattered windows, ice-encrusted throne, supernatural cold, winter sovereign's domain"},
        {"name": "battle_bg_cave_r5", "width": 1024, "height": 768, "prompt_xl": "JRPG underground cave battle background, stunning ice cavern, massive frozen stalactites and stalagmites of pure ice, translucent blue crystal walls refracting light into rainbows, frozen underground lake with dark depths visible beneath ice, cold blue atmosphere"},
        {"name": "battle_bg_r6", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG battle background, dramatic rocky canyon with towering red-brown sandstone cliffs, rushing river far below, ancient stone bridges spanning the gorge, distant mountain fortress perched on cliff edge, eagles soaring, rugged frontier landscape"},
        {"name": "battle_bg_r6_boss", "width": 1024, "height": 768, "prompt_xl": "JRPG boss battle background, crumbling medieval fortress under siege, burning towers with thick smoke billowing, broken rampart walls with rubble, catapult projectiles flying overhead, dramatic war-torn battlefield, fires and destruction, desperate last stand"},
        {"name": "battle_bg_cave_r6", "width": 1024, "height": 768, "prompt_xl": "JRPG underground cave battle background, vast dwarven mine complex, glittering ore veins of gold and mithril in rough-hewn stone walls, rusty minecart tracks leading into darkness, warm lantern light, wooden support beams, scattered mining tools and gems"},
        {"name": "battle_bg_r7", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG battle background, eerie dark swamp, gnarled twisted trees draped in hanging moss, murky stagnant water with lily pads, thick rolling fog, floating will-o-wisps glowing blue-green, decaying logs, ominous yet beautiful wetland, spooky atmosphere"},
        {"name": "battle_bg_r7_boss", "width": 1024, "height": 768, "prompt_xl": "JRPG boss battle background, haunted necromancer's graveyard, ancient crumbling tombstones and mausoleums, massive full moon casting silver light, skeletal hands rising from graves, thick purple-violet mist swirling, dead trees silhouetted, supernatural horror atmosphere"},
        {"name": "battle_bg_cave_r7", "width": 1024, "height": 768, "prompt_xl": "JRPG underground cave battle background, ancient catacombs, walls lined with neatly stacked skulls and bones, arched stone corridors, eerie green-flame torches casting sickly light, cobwebs, dark ritualistic symbols carved in stone, macabre underground ossuary"},
        {"name": "battle_bg_r8", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG battle background, serene Japanese-inspired cherry blossom garden, pink sakura petals falling gently, elegant stone pagoda, peaceful koi pond with stepping stones, curved wooden bridge, perfectly raked zen garden, tranquil eastern fantasy"},
        {"name": "battle_bg_r8_boss", "width": 1024, "height": 768, "prompt_xl": "JRPG boss battle background, burning Japanese temple complex, massive crimson torii gate crackling with demonic energy, blood-red sky, sacred grounds defiled by dark power, paper talismans burning, spiritual warfare between sacred and profane, dramatic eastern boss arena"},
        {"name": "battle_bg_cave_r8", "width": 1024, "height": 768, "prompt_xl": "JRPG underground cave battle background, natural hot spring cavern, steaming mineral pools in vivid blues and greens, warm golden-orange light from volcanic heat below, smooth water-carved stone formations, healing mist rising, peaceful yet dangerous underground onsen"},
        {"name": "battle_bg_r9", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG battle background, breathtaking floating islands in sky, fluffy clouds drifting below, ancient magical bridges of light connecting islands, ruins of sky castle covered in hanging gardens, waterfalls cascading into void, brilliant blue sky, Laputa-inspired aerial realm"},
        {"name": "battle_bg_r9_boss", "width": 1024, "height": 768, "prompt_xl": "JRPG boss battle background, collapsing ancient sky fortress breaking apart, massive dimensional rift tearing through reality, swirling energy vortex of purple and gold, stone fragments and debris floating in zero gravity, reality bending, cataclysmic aerial final battle"},
        {"name": "battle_bg_cave_r9", "width": 1024, "height": 768, "prompt_xl": "JRPG underground cave battle background, spectacular crystal cavern inside floating island, massive rainbow-refracting crystal formations, magical geodes cracked open showing brilliant gem interiors, prismatic light dancing across cave walls, ethereal otherworldly beauty"},
        {"name": "battle_bg_r10", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG battle background, hellish volcanic landscape, rivers of flowing bright orange lava, tall jagged obsidian rock spires, crimson sky filled with ash and embers, volcanic eruptions in distance, heat haze, infernal demon realm, Final Fantasy-style fire region"},
        {"name": "battle_bg_r10_boss", "width": 1024, "height": 768, "prompt_xl": "JRPG boss battle background, massive demon forge throne room, cascading lava waterfalls behind imposing dark throne, enormous chains and anvils scattered about, hellfire braziers lighting the chamber, obsidian floor cracked with lava veins, demonic runes glowing red, ultimate blacksmith domain"},
        {"name": "battle_bg_cave_r10", "width": 1024, "height": 768, "prompt_xl": "JRPG underground cave battle background, massive volcanic magma tube tunnel, river of bright flowing lava providing orange illumination, intense heat distortion in air, volcanic gas vents shooting steam, hardened black lava formations, suffocating underground inferno"},
        {"name": "battle_bg_r11", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG battle background, hauntingly beautiful dark ruins, crumbling gothic arches and flying buttresses under pale moonlight, glowing arcane magic circles on ground, climbing ivy and roses on ancient stone, scattered ancient tomes, dark academic atmosphere, romantic gothic fantasy"},
        {"name": "battle_bg_r11_boss", "width": 1024, "height": 768, "prompt_xl": "JRPG boss battle background, top of dark sorcerer's tower, massive arcane storm swirling above, intricate glowing ritual circle on floor, unstable dimensional portal crackling with energy, books and scrolls flying in magical wind, reality warping, apocalyptic magic unleashed"},
        {"name": "battle_bg_cave_r11", "width": 1024, "height": 768, "prompt_xl": "JRPG underground cave battle background, forgotten underground library carved into cave, towering bookshelves built into rock walls, magical scrolls and tomes floating in air, soft arcane blue light from enchanted crystals, ancient knowledge repository, scholarly dungeon"},
        {"name": "battle_bg_r12", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG final region battle background, imposing demon castle courtyard, massive dark stone walls and towers, blood-red tattered banners hanging, stone gargoyles perched on battlements, ominous swirling dark red-purple sky, bone-strewn ground, ultimate evil fortress, JRPG final dungeon exterior"},
        {"name": "battle_bg_r12_boss", "width": 1024, "height": 768, "prompt_xl": "JRPG ultimate final boss battle background, demon lord's colossal throne room, impossibly massive dark obsidian throne, towering hellfire pillars casting blood-red light, dimensional rifts tearing reality apart behind throne, floating debris, ultimate evil sanctum, end of the world atmosphere, most dramatic JRPG final battle setting"},
        {"name": "battle_bg_cave_r12", "width": 1024, "height": 768, "prompt_xl": "JRPG underground cave battle background, terrifying abyssal void, isolated floating stone platforms over infinite darkness below, streams of demonic red-purple energy flowing between platforms, distant tortured souls, edge of the underworld, most dangerous dungeon, final descent into darkness"}
    ],
    "interiors": [
        {"name": "interior_inn", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG inn interior scene, cozy medieval tavern and inn, warm wooden bar counter with tankards and bottles, oak barrels stacked behind, roaring stone fireplace with warm orange glow, wooden tables and chairs, hanging lanterns, comfortable beds visible upstairs landing, welcoming homey atmosphere, Secret of Mana style"},
        {"name": "interior_shop", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG item shop interior scene, bustling medieval merchant shop, wooden shelves lined with colorful potion bottles, swords and shields displayed on walls, glass display cases with jewelry and accessories, wooden merchant counter with scales and coins, warm candlelight, cluttered but organized, pixel art RPG shop"},
        {"name": "interior_church", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG church interior scene, serene medieval stone cathedral, magnificent stained glass windows casting colored light beams, ornate altar with golden holy symbol, wooden pews, high vaulted ceiling with stone arches, candelabras with flickering candles, peaceful sacred atmosphere, healing and salvation"},
        {"name": "interior_castle", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG castle throne room interior, grand royal hall, long red velvet carpet leading to ornate golden throne on raised dais, massive stone pillars, royal banners and tapestries on walls, stained glass windows, knight guards standing at attention, chandelier with hundreds of candles, majestic sovereign's domain"},
        {"name": "interior_house", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG cozy house interior scene, warm medieval cottage home, stone fireplace with crackling fire, wooden table set with bread and stew, comfortable bed with patchwork quilt, small bookshelf with worn books, herbs drying from ceiling beams, cat sleeping on rug, homey lived-in atmosphere"},
        {"name": "interior_cave", "width": 1024, "height": 768, "prompt_xl": "16-bit JRPG cave dungeon entrance interior, rough-hewn dark stone walls and ceiling, flickering wall torches providing orange light, old wooden support beams and mining props, scattered bones and debris on floor, dark passages leading deeper, ominous dripping water, adventure awaits in the darkness"}
    ]
}

for cat in ['tiles', 'battle_backgrounds', 'interiors']:
    print(f'{cat}: {len(PROMPTS[cat])} entries')

## 4. Load SDXL Pipeline

SDXL txt2img + img2img (shared components) for seamless tiles and scenes.

In [ ]:
from diffusers import (
    StableDiffusionXLPipeline,
    StableDiffusionXLImg2ImgPipeline,
    DPMSolverMultistepScheduler,
)

SDXL_HUB = 'stabilityai/stable-diffusion-xl-base-1.0'
LORA_HUB = 'nerijs/pixel-art-xl'

sdxl_cache = MODELS_DIR / 'stable-diffusion-xl-base-1.0'
model_path = str(sdxl_cache) if (sdxl_cache / 'model_index.json').exists() else SDXL_HUB

print(f'[pipeline] Loading SDXL txt2img + img2img ({DEVICE})...')
t0 = time.time()

kwargs = {'torch_dtype': DTYPE, 'use_safetensors': True, 'low_cpu_mem_usage': False}
if 'stabilityai' in model_path:
    kwargs['variant'] = 'fp16'

pipe_txt = StableDiffusionXLPipeline.from_pretrained(model_path, **kwargs)
pipe_txt.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe_txt.scheduler.config,
    algorithm_type='dpmsolver++',
    use_karras_sigmas=True,
)

try:
    pipe_txt.load_lora_weights(LORA_HUB, adapter_name='pixel_xl')
    pipe_txt.set_adapters(['pixel_xl'], adapter_weights=[0.8])
    print('[pipeline] pixel-art-xl LoRA loaded')
except Exception as e:
    print(f'[warn] LoRA failed: {e}')

pipe_txt = pipe_txt.to(DEVICE)
pipe_txt.enable_attention_slicing()

# img2img shares all components (no extra VRAM)
pipe_i2i = StableDiffusionXLImg2ImgPipeline(
    vae=pipe_txt.vae,
    text_encoder=pipe_txt.text_encoder,
    text_encoder_2=pipe_txt.text_encoder_2,
    tokenizer=pipe_txt.tokenizer,
    tokenizer_2=pipe_txt.tokenizer_2,
    unet=pipe_txt.unet,
    scheduler=pipe_txt.scheduler,
)

print(f'[pipeline] Ready in {time.time()-t0:.1f}s')
print_vram()

## 5. Seamless Tile Functions (Offset-Trick)

In [ ]:
def offset_image(image, dx, dy):
    """Roll image by (dx, dy) pixels, wrapping edges."""
    arr = np.array(image)
    arr = np.roll(arr, dy, axis=0)
    arr = np.roll(arr, dx, axis=1)
    return Image.fromarray(arr)


def generate_seamless_tile(prompt, negative='', gen_size=512, target=64,
                            steps=30, guidance=7.5, strength=0.5):
    """Offset-trick: gen base -> offset half -> inpaint seams -> offset back -> downscale."""
    full_prompt = f'seamless tileable texture, {prompt}, pixel art, top-down view, game tile'
    full_neg = negative or 'text, watermark, frame, border, non-tileable'
    half = gen_size // 2

    base = pipe_txt(
        prompt=full_prompt, negative_prompt=full_neg,
        width=gen_size, height=gen_size,
        num_inference_steps=steps, guidance_scale=guidance,
    ).images[0]

    offset = offset_image(base, half, half)

    inpainted = pipe_i2i(
        prompt=full_prompt, negative_prompt=full_neg,
        image=offset, strength=strength,
        num_inference_steps=max(10, int(steps * strength)),
        guidance_scale=guidance,
    ).images[0]

    seamless = offset_image(inpainted, half, half)
    tile = seamless.resize((target, target), Image.NEAREST)
    return tile


def render_tiled_preview(tile, grid=4):
    w, h = tile.size
    preview = Image.new('RGB', (w * grid, h * grid))
    for r in range(grid):
        for c in range(grid):
            preview.paste(tile, (c * w, r * h))
    return preview

## 6. Generate Seamless Tiles (12 tiles)

4 SDXL calls per tile. ~10-15s per tile, total ~2-3 minutes.

In [ ]:
entries = PROMPTS['tiles']
neg = PROMPTS['_meta']['negative_prompt_xl']
out_dir = OUTPUT_BASE / 'tiles'
out_dir.mkdir(parents=True, exist_ok=True)

tracker = ProgressTracker('env_tiles', OUTPUT_BASE)
remaining = sum(1 for e in entries if not tracker.is_done(e['name']))
print(f'Tiles: {len(entries)} total, {remaining} to generate')

for i, entry in enumerate(entries):
    name = entry['name']
    out_path = out_dir / f'{name}.png'

    if tracker.is_done(name) and out_path.exists():
        print(f'  [{i+1}/{len(entries)}] [skip] {name}')
        continue

    prompt = entry.get('prompt_xl', '')
    print(f'\n  [{i+1}/{len(entries)}] {name} -- {tracker.summary(len(entries))}')
    t0 = time.time()

    tile = generate_seamless_tile(prompt, neg, target=64)
    tile.save(str(out_path), 'PNG')

    preview = render_tiled_preview(tile)
    preview.save(str(out_dir / f'{name}_preview.png'), 'PNG')

    print(f'        saved: {name}.png + preview ({time.time()-t0:.1f}s)')
    tracker.mark_done(name)
    free_vram()

print('\nDone!')

## 7. Verify Tile Seamlessness

In [ ]:
import matplotlib.pyplot as plt

tiles_dir = OUTPUT_BASE / 'tiles'
previews = sorted(tiles_dir.glob('*_preview.png')) if tiles_dir.exists() else []

if previews:
    cols = 4
    rows = (len(previews) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    if rows == 1:
        axes = [axes]

    for i, p in enumerate(previews):
        r, c = i // cols, i % cols
        ax = axes[r][c] if rows > 1 else axes[c]
        ax.imshow(Image.open(p))
        ax.set_title(p.stem.replace('_preview', ''), fontsize=9)
        ax.axis('off')

    for i in range(len(previews), rows * cols):
        r, c = i // cols, i % cols
        ax = axes[r][c] if rows > 1 else axes[c]
        ax.axis('off')

    plt.suptitle('Seamless Tile Preview (4x4 tiled)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No tile previews found yet')

## 8. Generate Battle Backgrounds (36 images)

12 regions x 3 variants @ 1024x768, ~8s/img, total ~5 min.

In [ ]:
# Reduce LoRA weight for backgrounds
try:
    pipe_txt.set_adapters(['pixel_xl'], adapter_weights=[0.3])
    print('[pipeline] LoRA weight set to 0.3 for backgrounds')
except Exception:
    pass

entries = PROMPTS['battle_backgrounds']
neg = PROMPTS['_meta']['negative_prompt_xl']
out_dir = OUTPUT_BASE / 'battle_backgrounds'
out_dir.mkdir(parents=True, exist_ok=True)

tracker = ProgressTracker('env_battle_backgrounds', OUTPUT_BASE)
remaining = sum(1 for e in entries if not tracker.is_done(e['name']))
print(f'\nBattle Backgrounds: {len(entries)} total, {remaining} to generate')

for i, entry in enumerate(entries):
    name = entry['name']
    out_path = out_dir / f'{name}.png'

    if tracker.is_done(name) and out_path.exists():
        print(f'  [{i+1}/{len(entries)}] [skip] {name}')
        continue

    prompt = entry.get('prompt_xl', '')
    w, h = entry.get('width', 1024), entry.get('height', 768)

    print(f'\n  [{i+1}/{len(entries)}] {name} ({w}x{h}) -- {tracker.summary(len(entries))}')
    t0 = time.time()

    img = pipe_txt(
        prompt=prompt, negative_prompt=neg,
        width=w, height=h,
        num_inference_steps=30, guidance_scale=7.5,
    ).images[0]

    img.save(str(out_path), 'PNG')
    kb = out_path.stat().st_size / 1024
    print(f'        saved: {name}.png ({kb:.0f} KB, {time.time()-t0:.1f}s)')
    tracker.mark_done(name)
    free_vram()

# Restore LoRA weight
try:
    pipe_txt.set_adapters(['pixel_xl'], adapter_weights=[0.8])
except Exception:
    pass

print('\nDone!')

## 9. Generate Interior Scenes (6 images)

In [ ]:
try:
    pipe_txt.set_adapters(['pixel_xl'], adapter_weights=[0.3])
except Exception:
    pass

entries = PROMPTS['interiors']
neg = PROMPTS['_meta']['negative_prompt_xl']
out_dir = OUTPUT_BASE / 'interiors'
out_dir.mkdir(parents=True, exist_ok=True)

tracker = ProgressTracker('env_interiors', OUTPUT_BASE)
remaining = sum(1 for e in entries if not tracker.is_done(e['name']))
print(f'\nInteriors: {len(entries)} total, {remaining} to generate')

for i, entry in enumerate(entries):
    name = entry['name']
    out_path = out_dir / f'{name}.png'

    if tracker.is_done(name) and out_path.exists():
        print(f'  [{i+1}/{len(entries)}] [skip] {name}')
        continue

    prompt = entry.get('prompt_xl', '')
    w, h = entry.get('width', 1024), entry.get('height', 768)

    print(f'\n  [{i+1}/{len(entries)}] {name} ({w}x{h}) -- {tracker.summary(len(entries))}')
    t0 = time.time()

    img = pipe_txt(
        prompt=prompt, negative_prompt=neg,
        width=w, height=h,
        num_inference_steps=30, guidance_scale=7.5,
    ).images[0]

    img.save(str(out_path), 'PNG')
    kb = out_path.stat().st_size / 1024
    print(f'        saved: {name}.png ({kb:.0f} KB, {time.time()-t0:.1f}s)')
    tracker.mark_done(name)
    free_vram()

try:
    pipe_txt.set_adapters(['pixel_xl'], adapter_weights=[0.8])
except Exception:
    pass

print('\nDone!')

## 10. Review Battle Backgrounds

In [ ]:
bg_dir = OUTPUT_BASE / 'battle_backgrounds'
bgs = sorted(bg_dir.glob('*.png')) if bg_dir.exists() else []

if bgs:
    cols = 3
    rows = (len(bgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    if rows == 1:
        axes = [axes]

    for i, p in enumerate(bgs):
        r, c = i // cols, i % cols
        ax = axes[r][c] if rows > 1 else axes[c]
        ax.imshow(Image.open(p))
        ax.set_title(p.stem, fontsize=8)
        ax.axis('off')

    for i in range(len(bgs), rows * cols):
        r, c = i // cols, i % cols
        ax = axes[r][c] if rows > 1 else axes[c]
        ax.axis('off')

    plt.suptitle(f'Battle Backgrounds ({len(bgs)} images)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No battle backgrounds found yet')

## 11. Generate Manifest + Download

In [ ]:
import shutil

manifest = {}
for cat in ['tiles', 'battle_backgrounds', 'interiors', 'monsters', 'buildings', 'portraits', 'characters']:
    d = OUTPUT_BASE / cat
    if not d.exists():
        continue
    keys = sorted(f.stem for f in d.glob('*.png')
                  if not f.stem.startswith('_') and not f.stem.endswith('_preview'))
    if keys:
        manifest[cat] = keys

manifest_path = OUTPUT_BASE / 'manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Manifest:')
for cat, keys in manifest.items():
    print(f'  {cat}: {len(keys)}')

zip_path = '/content/environments_output'
shutil.make_archive(zip_path, 'zip', str(OUTPUT_BASE))
print(f'\nZip: {os.path.getsize(zip_path + ".zip") / 1024 / 1024:.1f} MB')

if ON_COLAB:
    from google.colab import files
    files.download(f'{zip_path}.zip')
else:
    print(f'Download: {zip_path}.zip')